# 🚀 RiskBricks — Complete System Setup

**One-click setup** for Databricks Summit demo. Run all cells to build the entire system from scratch.

| Phase | What | Est. Time |
|-------|------|-----------|
| 0 | Prerequisites (catalog, schemas, secrets) | 1 min |
| 1 | Data Ingestion (stocks, macros, portfolio, GDELT) | 30 min |
| 2 | Daily Refresh & Risk Analytics | 10 min |
| 3 | ML Pipeline (features, training, predictions) | 15 min |
| 4 | Forecasts & Portfolio Manager Outputs | 10 min |
| 5 | AI Agents (UC tools, agent, endpoint) | 25 min |
| 6 | App Deployment & Job Creation | 5 min |
| **Total** | **Full system ready** | **\~90 min** |

In [0]:
import time
from datetime import datetime, timedelta

# ── Widget ────────────────────────────────────────────────────────
dbutils.widgets.text("catalog", "riskbricks")
CATALOG = dbutils.widgets.get("catalog").strip()

# ── Auto-detect repo root ─────────────────────────────────────────
import os
_nb_path = dbutils.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = "/Workspace" + os.path.dirname(_nb_path)
NOTEBOOKS = f"{REPO_ROOT}/notebooks"

print(f"📂 Repo root:  {REPO_ROOT}")
print(f"📂 Notebooks:  {NOTEBOOKS}")
print(f"🗂️  Catalog:    {CATALOG}")

# ── Status tracker ────────────────────────────────────────────────
setup_log = []

def run_step(step_name, notebook_path, timeout=1800, params=None):
    """Run a notebook step with timing and error handling."""
    if params is None:
        params = {"catalog": CATALOG}
    elif "catalog" not in params:
        params["catalog"] = CATALOG

    print(f"\n{'─'*60}")
    print(f"▶ {step_name}")
    print(f"  Notebook: {notebook_path}")
    start = time.time()

    try:
        result = dbutils.notebook.run(notebook_path, timeout, params)
        elapsed = time.time() - start
        status = "✅ OK"
        setup_log.append((step_name, status, f"{elapsed:.0f}s", ""))
        print(f"  {status} ({elapsed:.0f}s)")
        return True
    except Exception as e:
        elapsed = time.time() - start
        status = "❌ FAILED"
        error_msg = str(e)[:200]
        setup_log.append((step_name, status, f"{elapsed:.0f}s", error_msg))
        print(f"  {status} ({elapsed:.0f}s)")
        print(f"  Error: {error_msg}")
        return False

setup_start = time.time()
print(f"\n🚀 Setup started at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Phase 0 — Prerequisites
Create the Unity Catalog, schemas, and verify environment.

In [0]:
print("🏗️  Creating catalog and schemas...")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"USE CATALOG {CATALOG}")

schemas = ["bronze", "silver", "gold", "agent_tools", "agents", "models", "pipelines"]
for s in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{s}")
    print(f"  ✅ {CATALOG}.{s}")

# Verify FRED API key (optional — some notebooks have fallback)
try:
    fred_key = dbutils.secrets.get(scope="riskbricks", key="fred-api-key")
    print(f"\n🔑 FRED API key: found in secrets")
except Exception:
    print(f"\n⚠️  FRED API key not in secrets — notebooks will use fallback")

setup_log.append(("Phase 0: Prerequisites", "✅ OK", "< 1s", ""))
print("\n✅ Phase 0 complete")

## Phase 1 — Data Ingestion (\~30 min)
Load 12 years of historical stock prices, macro indicators, portfolio config, and GDELT events.

In [0]:
# Step 1.1: Historical stocks + FRED macro (12 years) → bronze
run_step(
    "1.1 Ingest historical stocks & macros (2015-present)",
    f"{NOTEBOOKS}/ingestion/stocks/ingest_stocks_and_macros_data",
    timeout=3600
)

# Step 1.2: Bronze → Silver → Gold transforms for stocks/macros
run_step(
    "1.2 Bronze → Gold stocks & macros pipeline",
    f"{NOTEBOOKS}/ingestion/stocks/bronze_to_gold_daily_stocks_macros",
    timeout=2400
)

# Step 1.3: Portfolio setup (company_universe, managers, holdings)
run_step(
    "1.3 Setup multi-manager portfolios",
    f"{NOTEBOOKS}/ingestion/portfolio/ingest_setup_multi_manager_portfolios",
    timeout=600
)

# Step 1.4: GDELT events (geopolitical)
run_step(
    "1.4 Ingest GDELT events",
    f"{NOTEBOOKS}/jobs/daily_gdelt_refresh",
    timeout=1800
)

print("\n✅ Phase 1 complete — Bronze + Gold foundation tables populated")

## Phase 2 — Daily Refresh & Risk Analytics (\~10 min)
Run the daily data refresh pipeline and compute portfolio risk metrics.

In [0]:
# Step 2.1: Daily data refresh (incremental stock prices → silver → gold updates)
run_step(
    "2.1 Daily data refresh (stock prices → risk metrics)",
    f"{NOTEBOOKS}/jobs/daily_data_refresh",
    timeout=1200
)

# Step 2.2: Full risk analytics (VaR, stress tests, factor exposures)
run_step(
    "2.2 Risk analytics (VaR, stress tests, factors)",
    f"{NOTEBOOKS}/gold/analytics/create_risk_analytics",
    timeout=1200
)

print("\n✅ Phase 2 complete — Risk metrics and analytics computed")

## Phase 3 — ML Pipeline (\~15 min)
Ingest RSS/FRED, compute features, train ensemble model, generate ML predictions.

In [0]:
# Step 3.1: ML data ingestion (RSS news, FRED macro, technical indicators,
#           sector features, AI sentiment, ML feature assembly, ML predictions)
run_step(
    "3.1 ML data ingestion (RSS + FRED + features + predictions)",
    f"{NOTEBOOKS}/ingestion/ml_data_ingestion",
    timeout=2400
)

# Step 3.2: Train & register ensemble model (LightGBM + RF + GBM)
run_step(
    "3.2 Train & register ensemble model",
    f"{NOTEBOOKS}/training/train_register_ensemble_model",
    timeout=1200,
    params={"catalog": CATALOG, "lookback_days": "30"}
)

print("\n✅ Phase 3 complete — ML models trained and registered")

## Phase 4 — Forecasts & Portfolio Outputs (\~10 min)
Build forecast features, generate predictions, evaluate accuracy, and create portfolio manager views.

In [0]:
today = datetime.now().strftime("%Y-%m-%d")
thirty_days_ago = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")

# Step 4.1: Build forecast features
run_step(
    "4.1 Build forecast features",
    f"{NOTEBOOKS}/ingestion/forecast/build_forecast_features_daily",
    timeout=900,
    params={"catalog": CATALOG, "start_date": thirty_days_ago, "end_date": today}
)

# Step 4.2: Train forecast model (Ridge baseline)
run_step(
    "4.2 Train forecast model (Ridge)",
    f"{NOTEBOOKS}/gold/forecast/train_forecast_model",
    timeout=600,
    params={"catalog": CATALOG, "start_date": thirty_days_ago, "end_date": today}
)

# Step 4.3: Generate stock forecasts (1d + 15d)
run_step(
    "4.3 Generate stock forecasts",
    f"{NOTEBOOKS}/gold/forecast/generate_stock_forecasts",
    timeout=600,
    params={"catalog": CATALOG, "as_of_date": today}
)

# Step 4.4: Evaluate forecast accuracy
run_step(
    "4.4 Evaluate stock forecasts",
    f"{NOTEBOOKS}/gold/forecast/evaluate_stock_forecasts",
    timeout=600,
    params={"catalog": CATALOG, "forecast_date": today}
)

# Step 4.5: Build portfolio manager outputs (decision signals, risk-adjusted view, etc.)
run_step(
    "4.5 Build portfolio manager outputs",
    f"{NOTEBOOKS}/gold/analytics/build_portfolio_manager_outputs",
    timeout=600,
    params={"catalog": CATALOG, "start_date": today, "end_date": today}
)

print("\n✅ Phase 4 complete — Forecasts generated and portfolio outputs ready")

## Phase 5 — AI Agents (\~25 min)
Register Unity Catalog tools, create the multi-agent system, and deploy the serving endpoint.

In [0]:
# Step 5.1: Register all UC tool functions (11 functions in agent_tools schema)
run_step(
    "5.1 Register UC tools (11 functions)",
    f"{NOTEBOOKS}/agents/01_register_uc_tools",
    timeout=600
)

# Step 5.2: Create agent, test, log to MLflow, register in UC
run_step(
    "5.2 Create & test agent (MLflow + UC registration)",
    f"{NOTEBOOKS}/agents/02_create_agent",
    timeout=1200
)

# Step 5.3: Deploy serving endpoint (takes ~15-20 min to provision)
run_step(
    "5.3 Deploy agent serving endpoint",
    f"{NOTEBOOKS}/agents/03_deploy_agent",
    timeout=2400
)

print("\n✅ Phase 5 complete — AI agents deployed and serving")

## Phase 6 — App Deployment & Jobs (\~5 min)
Deploy the Streamlit app and create scheduled jobs for daily operations.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# ── Deploy Streamlit App ──────────────────────────────────────────
print("🚀 Deploying Streamlit app...")
app_name = "riskbricks-app"
app_source = f"{REPO_ROOT}/app"

try:
    try:
        existing = w.apps.get(app_name)
        print(f"  ℹ️  App '{app_name}' already exists — creating new deployment")
        deployment = w.apps.deploy(app_name, source_code_path=app_source)
        print(f"  ✅ App redeployed: {app_name}")
    except Exception:
        app = w.apps.create(
            name=app_name,
            description="RiskBricks Portfolio Risk Analytics"
        )
        print(f"  ✅ App created: {app_name}")
    setup_log.append(("6.1 Deploy Streamlit app", "✅ OK", "", ""))
except Exception as e:
    error_msg = str(e)[:200]
    print(f"  ❌ App deployment failed: {error_msg}")
    setup_log.append(("6.1 Deploy Streamlit app", "❌ FAILED", "", error_msg))

# ── Create Daily Jobs ─────────────────────────────────────────────
print("\n📋 Creating daily scheduled jobs...")

from databricks.sdk.service.jobs import (
    Task, NotebookTask, CronSchedule, TaskDependency,
)

job_configs = [
    {
        "name": "RiskBricks — Daily Data Refresh",
        "tasks": [
            {"key": "stock_refresh", "notebook": f"{NOTEBOOKS}/jobs/daily_data_refresh"},
            {"key": "gdelt_refresh", "notebook": f"{NOTEBOOKS}/jobs/daily_gdelt_refresh"},
        ],
        "cron": "0 0 23 * * ?",  # 11 PM UTC = 6 PM ET
    },
    {
        "name": "RiskBricks — ML Pipeline",
        "tasks": [
            {"key": "ml_ingestion", "notebook": f"{NOTEBOOKS}/ingestion/ml_data_ingestion"},
            {"key": "forecast_features", "notebook": f"{NOTEBOOKS}/ingestion/forecast/build_forecast_features_daily"},
            {"key": "generate_forecasts", "notebook": f"{NOTEBOOKS}/gold/forecast/generate_stock_forecasts", "depends": ["forecast_features"]},
            {"key": "portfolio_outputs", "notebook": f"{NOTEBOOKS}/gold/analytics/build_portfolio_manager_outputs", "depends": ["generate_forecasts"]},
        ],
        "cron": "0 30 23 * * ?",  # 11:30 PM UTC = 6:30 PM ET
    },
]

for jc in job_configs:
    try:
        tasks = []
        for t in jc["tasks"]:
            depends = [TaskDependency(task_key=d) for d in t.get("depends", [])]
            task = Task(
                task_key=t["key"],
                notebook_task=NotebookTask(
                    notebook_path=t["notebook"],
                    base_parameters={"catalog": CATALOG},
                    source="WORKSPACE",
                ),
                depends_on=depends if depends else None,
            )
            tasks.append(task)

        # Check if job already exists
        existing_jobs = w.jobs.list(name=jc["name"])
        job_exists = False
        for ej in existing_jobs:
            if ej.settings.name == jc["name"]:
                job_exists = True
                print(f"  ℹ️  Job '{jc['name']}' already exists (ID: {ej.job_id})")
                break

        if not job_exists:
            job = w.jobs.create(
                name=jc["name"],
                tasks=tasks,
                schedule=CronSchedule(
                    quartz_cron_expression=jc["cron"],
                    timezone_id="America/New_York",
                    pause_status="PAUSED",
                ),
                max_concurrent_runs=1,
                tags={"project": "riskbricks", "env": "demo"},
            )
            print(f"  ✅ Created job: {jc['name']} (ID: {job.job_id}) [PAUSED]")
    except Exception as e:
        print(f"  ⚠️  Job '{jc['name']}' creation note: {str(e)[:150]}")

setup_log.append(("6.2 Create scheduled jobs", "✅ OK", "", ""))
print("\n✅ Phase 6 complete — App deployed and jobs created (paused)")

## 🎉 Setup Summary

In [0]:
total_elapsed = time.time() - setup_start
minutes = int(total_elapsed // 60)
seconds = int(total_elapsed % 60)

print("=" * 70)
print("🎉 RISKBRICKS SETUP COMPLETE")
print("=" * 70)
print(f"\nTotal time: {minutes}m {seconds}s")
print(f"Completed:  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n{'Step':<55} {'Status':<10} {'Time':<8}")
print("─" * 75)
for step_name, status, elapsed, error in setup_log:
    print(f"{step_name:<55} {status:<10} {elapsed:<8}")
    if error:
        print(f"  └─ {error[:70]}")

# Count successes/failures
successes = sum(1 for _, s, _, _ in setup_log if "✅" in s)
failures = sum(1 for _, s, _, _ in setup_log if "❌" in s)

print(f"\n{'─'*75}")
print(f"Results: {successes} passed, {failures} failed out of {len(setup_log)} steps")

if failures == 0:
    print(f"""
╔═════════════════════════════════════════════════════════════════╗
║  🚀 SYSTEM READY FOR DEMO                                       ║
║                                                                  ║
║  Catalog:   {CATALOG:<50} ║
║  App:       riskbricks-app                                       ║
║  Endpoint:  riskbricks-supervisor-agent                          ║
║  Tables:    ~40 across bronze/silver/gold                        ║
║  UC Tools:  11 functions in agent_tools                          ║
║  ML Model:  riskbricks.models.stock_forecast_ensemble            ║
║                                                                  ║
║  Try: "What is the risk in my portfolio?"                        ║
╚═════════════════════════════════════════════════════════════════╝
""")
else:
    print(f"\n⚠️  {failures} step(s) failed. Review errors above and re-run failed steps individually.")